In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets sentencepiece pandas textstat rouge-score bert-score sacrebleu matplotlib


In [ ]:
import gc
import re
import torch
import pandas as pd
import matplotlib.pyplot as plt

from textstat import flesch_kincaid_grade, automated_readability_index
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Hugging Face login.
# Some models require accepting the model license on Hugging Face first.
# In Colab: left sidebar → Secrets → add HF_TOKEN with your Hugging Face READ token.

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN found. If this model is gated, loading will fail until you add HF_TOKEN.")


In [ ]:
# Change this if your file name is different
DATA_PATH = "/content/pilot_20_baseline_eval.csv"

df = pd.read_csv(DATA_PATH)

print("Columns found:")
print(df.columns.tolist())
print("\nNumber of rows:", len(df))

df.head()

In [ ]:
# Required input columns
required_columns = ["source_text", "simple_text"]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Input CSV is valid.")

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_LABEL = "Llama 3.2 3B Instruct"

OUTPUT_COLUMN = "llama32_output"
FKGL_COLUMN = "llama32_fkgl"
ARI_COLUMN = "llama32_ari"
ROUGE_COLUMN = "llama32_rougeL"
BERTSCORE_COLUMN = "llama32_bertscore_f1"

In [ ]:
# Create model-specific output columns if they don't already exist
columns_to_create = [
    OUTPUT_COLUMN,
    FKGL_COLUMN,
    ARI_COLUMN,
    ROUGE_COLUMN,
    BERTSCORE_COLUMN
]

for col in columns_to_create:
    if col not in df.columns:
        df[col] = None

print("Model-specific columns are ready.")

In [ ]:
from transformers import BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama tokenizer has no pad token by default; set it to eos_token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()

print("Model loaded in 4-bit.")
print("CUDA available:", torch.cuda.is_available())


In [ ]:
def build_messages(source_text: str):
    return [
        {"role": "system", "content": "You are a careful legal simplification assistant."},
        {"role": "user", "content": f"""Simplify the Sri Lankan legal provision below into plain English.

Rules:
- Preserve the legal meaning as closely as possible.
- Keep SHALL, MAY, SHALL NOT.
- Keep numbers, dates, fines, punishments, conditions, exceptions, and legal roles.
- Do not add examples, commentary, or legal advice.
- Do not include any preamble, introduction, or label such as "Here is the simplified version" or "Plain English:" -- output only the simplified provision itself.
- Write the answer as plain prose using full sentences. Break multi-part conditions or punishments into separate short sentences instead of one long sentence. Do not use bullet points, numbered lists, or headings.

Provision:
{source_text}

Plain English:"""}
    ]


In [ ]:
# Safety net: strip a leading meta-commentary preamble if the model adds one
# anyway (e.g. "Here's a simplified version:") despite the prompt rule against it.
# Applied after decoding, before the text is scored or saved.
PREAMBLE_RE = re.compile(
    r"^(here'?s|here is|sure[,!]?|certainly[,!]?|simplified version|plain english).*?:\s*",
    re.IGNORECASE
)

def strip_preamble(text: str) -> str:
    return PREAMBLE_RE.sub("", text.strip(), count=1).strip()


In [ ]:
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 192

def generate_output(source_text: str) -> str:
    messages = build_messages(source_text)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][input_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    text = strip_preamble(text)

    del inputs, outputs, generated_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return text


In [ ]:
# Optional quick test.
# Keep this False when using Colab Free, because testing first generates extra outputs and wastes RAM/time.
RUN_QUICK_TEST = False

if RUN_QUICK_TEST:
    for i in range(min(1, len(df))):
        source = str(df.loc[i, "source_text"])
        gold = str(df.loc[i, "simple_text"]) if pd.notna(df.loc[i, "simple_text"]) else ""

        pred = generate_output(source)

        print("ROW:", i)
        print("\nSOURCE:\n", source)
        print("\nGOLD SIMPLE TEXT:\n", gold)
        print("\nMODEL OUTPUT:\n", pred)
        print("\n" + "=" * 100 + "\n")
else:
    print("Quick test skipped. Set RUN_QUICK_TEST=True if you want to test one row first.")


In [ ]:
# RAM-safe generation loop with checkpoint saving.
# This saves after every row, so if Colab crashes you can rerun and continue.
import os

CHECKPOINT_PATH = "/content/llama32_generation_checkpoint.csv"

if os.path.exists(CHECKPOINT_PATH):
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH)
    if OUTPUT_COLUMN in checkpoint_df.columns and len(checkpoint_df) == len(df):
        df[OUTPUT_COLUMN] = checkpoint_df[OUTPUT_COLUMN]
        print(f"Loaded existing checkpoint: {CHECKPOINT_PATH}")
    else:
        print("Checkpoint found but does not match this dataset. Starting fresh.")
else:
    print("No checkpoint found. Starting fresh.")

for idx, text in df["source_text"].fillna("").astype(str).items():
    existing = df.loc[idx, OUTPUT_COLUMN] if OUTPUT_COLUMN in df.columns else None

    if isinstance(existing, str) and existing.strip():
        print(f"Skipping row {idx + 1}/{len(df)} - already generated")
        continue

    pred = generate_output(text)
    df.loc[idx, OUTPUT_COLUMN] = pred

    # Save after every row
    df.to_csv(CHECKPOINT_PATH, index=False)
    print(f"Done {idx + 1}/{len(df)} - checkpoint saved")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Generation complete.")


In [ ]:
def safe_fkgl(text):
    text = str(text).strip()
    if not text:
        return None
    try:
        return float(flesch_kincaid_grade(text))
    except Exception:
        return None

def safe_ari(text):
    text = str(text).strip()
    if not text:
        return None
    try:
        return float(automated_readability_index(text))
    except Exception:
        return None

df[FKGL_COLUMN] = df[OUTPUT_COLUMN].apply(safe_fkgl)
df[ARI_COLUMN] = df[OUTPUT_COLUMN].apply(safe_ari)

df[[OUTPUT_COLUMN, FKGL_COLUMN, ARI_COLUMN]].head()

In [ ]:
# ROUGE-L against human gold simplification
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

rouge_scores = []

for pred, ref in zip(df[OUTPUT_COLUMN].fillna("").astype(str), df["simple_text"].fillna("").astype(str)):
    pred = pred.strip()
    ref = ref.strip()

    if not pred or not ref:
        rouge_scores.append(None)
    else:
        score = scorer.score(ref, pred)
        rouge_scores.append(float(score["rougeL"].fmeasure))

df[ROUGE_COLUMN] = rouge_scores

df[[OUTPUT_COLUMN, "simple_text", ROUGE_COLUMN]].head()

In [ ]:
# BERTScore against human gold simplification.
# IMPORTANT: unload the LLM before BERTScore. BERTScore loads another transformer model.
try:
    del model
except NameError:
    pass

try:
    del tokenizer
except NameError:
    pass

try:
    del processor
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

preds = df[OUTPUT_COLUMN].fillna("").astype(str).tolist()
refs = df["simple_text"].fillna("").astype(str).tolist()

valid_indices = []
valid_preds = []
valid_refs = []

for i, (pred, ref) in enumerate(zip(preds, refs)):
    if pred.strip() and ref.strip():
        valid_indices.append(i)
        valid_preds.append(pred)
        valid_refs.append(ref)

bert_f1_scores = [None] * len(df)

if valid_preds:
    P, R, F1 = bertscore_score(
        valid_preds,
        valid_refs,
        lang="en",
        batch_size=4,
        verbose=False,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    for idx, score in zip(valid_indices, F1):
        bert_f1_scores[idx] = float(score)

df[BERTSCORE_COLUMN] = bert_f1_scores

df[[OUTPUT_COLUMN, "simple_text", BERTSCORE_COLUMN]].head()


In [ ]:
# Create a clean result table without dataset-analysis metadata columns
keep_cols = []

# Keep useful identifiers only if they exist
for col in ["example_id", "act_name", "section_id", "unit_type", "split"]:
    if col in df.columns:
        keep_cols.append(col)

keep_cols += [
    "source_text",
    "simple_text",
    OUTPUT_COLUMN,
    FKGL_COLUMN,
    ARI_COLUMN,
    ROUGE_COLUMN,
    BERTSCORE_COLUMN
]

result_df = df[keep_cols].copy()

print("Clean result table preview:")
result_df.head(10)

In [ ]:
def avg_of_column(dataframe, col_name):
    vals = pd.to_numeric(dataframe[col_name], errors="coerce")
    vals = vals.dropna()
    return None if len(vals) == 0 else round(float(vals.mean()), 3)

summary = {
    "Model": MODEL_LABEL,
    "Rows": len(result_df),
    "Avg FKGL": avg_of_column(result_df, FKGL_COLUMN),
    "Avg ARI": avg_of_column(result_df, ARI_COLUMN),
    "Avg ROUGE-L": avg_of_column(result_df, ROUGE_COLUMN),
    "Avg BERTScore F1": avg_of_column(result_df, BERTSCORE_COLUMN),
}

summary_df = pd.DataFrame([summary])
summary_df

In [ ]:
OUT_PATH = "/content/pilot_20_baseline_eval_with_llama32_checked_clean.csv"
result_df.to_csv(OUT_PATH, index=False)

print(f"Saved clean result CSV to: {OUT_PATH}")

In [ ]:
# Create summary table image for screenshot/report
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.axis("off")

plt.title(f"{MODEL_LABEL} - Baseline Evaluation Summary", fontsize=14, pad=14)

table = ax.table(
    cellText=summary_df.values,
    colLabels=summary_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.6)

IMAGE_PATH = "/content/llama32_summary_table.png"
plt.savefig(IMAGE_PATH, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved summary table image to: {IMAGE_PATH}")

In [ ]:
# Push both the per-model result CSV and the summary-table PNG to your browser's
# Downloads folder. /content/ is on the Colab VM's disk and disappears when the
# runtime resets, so this is the step that actually gets the files onto your machine.
from google.colab import files

files.download(OUT_PATH)
files.download(IMAGE_PATH)

In [ ]:
# Optional: clear memory
for name in ["model", "tokenizer", "processor"]:
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleared.")
